Ideas:
Time series analysis of native and invasive species, on a map 
<list>
    <p>- Can we visualize range contraction of native species using only inaturalist data? </p>
   <p> - Can we visualize elevation changes over time of certain species (birds)</p>
  <p>  - Can we visualize range expansion of nonnative species using only inaturalist data?</p>
</list>
How has the amount of native species and introduced species observations changed with proportion to the total amount that year, per year?
<br>
Introduced vs not introduced species prevalence, year by year<br>
How many species have gone extinct in the timeframe that INaturalist has operated?<br>
What regions have the highest biodiversity?<br>
what regions have the highest/lowest nonnative biodiversity? How has it changed over time? per island (use coordinates that fall within bounds that box an island)<br>
sort into groups based on island coordinates<br>
check every other coordinate and calculate how many unique species are within 1km of the coordinates<br>
what regions have the highest/lowest native biodiversity? How has it changed over time?





In [1]:
import requests
import pandas as pd
import geopandas as gpd
import matplotlib as plt
import numpy as np

<h1>Coordinates</h1>

In [4]:
url="https://api.open-meteo.com/v1/elevation?latitude=20.7878933107&longitude=-156.4726734198"
print(requests.get(url).json())


{'elevation': [0.0]}


In [5]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Filter the dataframe for the wolf snail
wolfsnail = data.loc[data["taxon_name"] == "Euglandina rosea"]

# Assuming 'location' contains a comma-separated string of "longitude,latitude"
# Convert the 'location' column to a list of Points
wolfsnail["geometry"] = wolfsnail["location"].apply(lambda x: Point(map(float, x.split(","))))

# Create a GeoDataFrame
geowolf = gpd.GeoDataFrame(wolfsnail, geometry="geometry")
geowolf["geometry"] = geowolf["geometry"].apply(lambda x: x.__class__(x.x, -x.y))
# Optionally, set the CRS if necessary (e.g., WGS84)
# geowolf.set_crs("EPSG:4326", inplace=True)

# Plot the data
geowolf.plot()

geowolf.explore()

NameError: name 'data' is not defined

<h1>Extinct Species:</h1>

In [4]:
data = pd.read_csv(r"INaturalistDataset.csv")

#returns the name of each extinct species, and the number of occurances.
extinct_data_first = data.sort_values(by="rank_extinct", axis=0, ascending=False)
extinct_data = data.loc[data["rank_extinct"]==True]
print(extinct_data["taxon_name"].value_counts())







taxon_name
Carelia cochlea                 7
Amastra morticina               3
Achatinella bulimoides rosea    2
Newcombia gagei                 2
Sturanya stokesii               2
Carelia necra                   2
Carelia evelynae                2
Achatinella abbreviata          2
Carelia dolei                   2
Carelia bicolor                 1
Leptachatina isthmica           1
Amastra forbesi                 1
Sturanya juddii                 1
Geograpsus severnsi             1
Ammonitida                      1
Mareleptopoma kenneyi           1
Carelia                         1
Otodus megalodon                1
Loxops wolstenholmei            1
Auriculella uniplicata          1
Achatinella livida              1
Partulina crassa                1
Carelia lirata                  1
Name: count, dtype: int64


In [ ]:
lizard = data.loc[data["observed_on"][0:4] in ()"Achatinella livida"]
lizard.head()

,taxon_name,id,location,positional_accuracy,observed_on,species_guess,taxon_rank,preferred_common_name,photo_url,copyright,conservation_status,establishment_means,location_obscured,rank_extinct
199708,Achatinella livida,191639662,"21.5425005852,-158.1956579316",47398.0,1893-12-31,Achatinella livida,species,NaN,https://inaturalist-open-data.s3.amazonaws.com...,"(c) Sean Werle, some rights reserved (CC BY-NC)",critically imperiled,endemic,True,True


<h1>Data Cleaning</h1>
Use preferred common name instead of species guess, as species guess is sometimes in a foregin language and does not translate properly.
<list>

standardize the taxon name, preferred common name in capitals<br>
Conservation status, Establishment Means, and observed_on- None to 'not_listed'<br>
Some observations may not have an observation date, however that does not mean that the upload date is accurate of the observation
<br>
Though there are observations as early as 1858, these are usually specimens that were wild at the time of observation, and not in captivity. As such, they will be included, though the data may not be very useful. 
</list>

Notes: Kevin Faccenda has over 33,532 total observations in hawaii. That seems a little bit biased.

<h1>How have observations of species changed over time></h1><p>
good idea: for each unique species, find their change in observations every year. graph how many species fall into %increase and %decrease intervals (5%) including 0% as their extinct, more than that goes up to the highest increase in a species. <br>
-only taxa with Genus species, or Genus species subspecies, or Genus species x species.<br>
-separate each 5% chucnk into both native/endemic and introduced.
-statistical analysis of whether these changes are due to limitations of the site or are real changes.  
</P>
Need: Dataframe of only native, endemic, or introduced species (not 'Not specified' establishment means), and species specified<br>
new dataframe: columns are yearly avg % change, rows are species 
Second graph: compare total %change from start to finish.

Need to normalize the annual change data based on how many total observations were made that year compared to total.

so avg = (amount previoius year/amount that year) but I also need to factor in (total prev/total that year) to get a change in total observations to normalize it

so it should be 100* (1- (amount prev. year/amount total prev year)/(amount year/amount total year))


<h3>Data Cleaning an Preparation for Population Trends</h3>
<h4>Step 1: Define helpful methods</h4>
- Helper method to retrieve the total per species per year<br>
- Combine the data of subspecies with the data of the species<br>


Need to: <br>incorporate establishment means and combine between subspecies and species, perhaps by a ranked dictionary comparison<br>
Find earliest years and make datafram include every year.

<h4>Step 2: Generate the new dataset of annual observation counts</h4>

In [47]:
graph1.head()

,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,taxon_name,id,location,positional_accuracy,observed_on,species_guess,taxon_rank,preferred_common_name,photo_url,copyright,conservation_status,establishment_means,location_obscured,rank_extinct,Unnamed: 0.1,Unnamed: 0,index
0,296592,296592,296592,Chelonia mydas,144336308,"19.1202886017,-155.4897906925",17.0,2022-12-10,Green Sea Turtle,species,Green Sea Turtle,https://static.inaturalist.org/photos/24770561...,"(c) Margaret Rousser, all rights reserved",vulnerable,native,True,False,NaN,NaN,NaN
1,226645,226645,226645,Chelonia mydas,179388553,"19.8536480907,-155.9016212178",6.0,2023-08-14,Green Sea Turtle,species,Green Sea Turtle,https://inaturalist-open-data.s3.amazonaws.com...,"(c) cello caruso-turiello, some rights reserve...",vulnerable,native,True,False,NaN,NaN,NaN
2,67808,67808,67808,Chelonia mydas,245841999,"19.9099035949,-155.9406315087",6.0,2024-10-05,Green Sea Turtle,species,Green Sea Turtle,https://inaturalist-open-data.s3.amazonaws.com...,"(c) kceraso, some rights reserved (CC BY-NC)",vulnerable,native,True,False,24068.0,NaN,NaN
3,226606,226606,226606,Chelonia mydas,179394518,"19.1064608317,-155.5078441276",230.0,2012-02-11,Green Sea Turtle,species,Green Sea Turtle,https://inaturalist-open-data.s3.amazonaws.com...,"(c) Lindley Ashline, some rights reserved (CC ...",vulnerable,native,True,False,NaN,NaN,NaN
4,226570,226570,226570,Chelonia mydas,179429208,"19.7482766189,-155.8006838129",242.0,2023-08-20,Green Sea Turtle,species,Green Sea Turtle,https://inaturalist-open-data.s3.amazonaws.com...,"(c) Cemone Hedges, some rights reserved (CC BY...",vulnerable,native,True,False,NaN,NaN,NaN


In [34]:
print(taxon_list)

Index(['Chelonia mydas', 'Paroaria coronata', 'Geopelia striata',
       'Metrosideros polymorpha', 'Phelsuma laticauda', 'Anolis sagrei',
       'Acridotheres tristis', 'Pluvialis fulva', 'Ardea ibis',
       'Scaevola taccada',
       ...
       'Hesperomannia arborescens', 'Jatropha curcas', 'Heliconia champneiana',
       'Isodendrion pyrifolium', 'Omorgus suberosus', 'Palea steindachneri',
       'Pinus thunbergii', 'Rhinacanthus nasutus', 'Bucculatrix thurberiella',
       'Triticum aestivum'],
      dtype='object', name='taxon_name', length=3875)


In [1]:
print(data.loc[(data["taxon_name"]=="Chelonia mydas") & (data["observed_on"].str[0:4]=='1969')]["id"])
#83100326 needs to be removed
#87491794 needs to be changed or date removed
#67606213 wrong date (2021)
#67438799 wrong date 2021
#67274768 same
#67237365 2020
#67121414 2020
#67121403 2020
#66910344 2020
#64205729 2020
#64202195 2020
#28344591 2019
#28344484 2019
#28344378 2019
#28342827 2019
#21933776 2019

#92117095
#102667950

SyntaxError: invalid syntax (1826074500.py, line 1)

<h3>Obtain a set of annual totals across all taxa for the purpose of data standardization</h3>

In [10]:
totals = totals.reset_index()  # Reset index and assign it back
# Compute column sums
sum_row = totals.sum(numeric_only=True)  # Sum only numeric columns

# Insert sums as the first row
totals.loc[-1] = sum_row 
totals = totals.sort_index().reset_index(drop=True)  # Shift indices properly
totals.iloc[0,0] = "All Taxa"
print(totals)
totals.to_csv("annualtotals.csv")

                    taxon_name  total_observations     2025      2024  \
0                     All Taxa            366814.0  12006.0  108525.0   
1               Chelonia mydas              5081.0    115.0    1212.0   
2            Paroaria coronata              4676.0    128.0    1322.0   
3             Geopelia striata              4346.0    171.0    1324.0   
4      Metrosideros polymorpha              4399.0    173.0    1245.0   
...                        ...                 ...      ...       ...   
3871       Palea steindachneri                 1.0      0.0       1.0   
3872          Pinus thunbergii                 1.0      0.0       1.0   
3873      Rhinacanthus nasutus                 1.0      0.0       1.0   
3874  Bucculatrix thurberiella                 1.0      0.0       0.0   
3875         Triticum aestivum                 1.0      0.0       0.0   

         2023      2022     2021    2020    2019    2018  ...  1974  1973  \
0     72464.0  102716.0  50880.0  1779.0  2209

<h3>Finally: Compute the standardized %change year to year and graph.</h3>
<br>
My formula for annual population change:


$$\text{Proportion in year} = \frac{\text{Taxa Observations in Year}}{\text{Total Taxa Observed in Year}}$$
$$\text{Percent Change} = \left( \frac{\text{Year Proportion} - \text{Previous Year Proportion}}{\text{Previous Year Proportion}} \right) \times 100$$


In [31]:
totals.head()

,total_observations,establishment_means,2025,2024,2023,2022,2021,2020,2019,2018,...,1974,1973,1972,1971,1969,1961,1947,1946,1933,1893
taxon_name,,,,,,,,,,,,,,,,,,,,,
Chelonia mydas,5081,NaN,115,1212,1000,1321,743,50,95,121,...,0,0,0,0,1,0,0,0,0,0
Paroaria coronata,4676,NaN,128,1322,970,1226,750,11,38,48,...,0,0,0,0,0,0,0,0,0,0
Geopelia striata,4346,NaN,171,1324,951,1065,610,11,42,33,...,0,0,0,0,0,0,0,0,0,0
Metrosideros polymorpha,4399,NaN,173,1245,821,1189,623,18,55,28,...,0,0,2,0,0,0,0,0,0,0
Phelsuma laticauda,5353,NaN,130,1538,1199,1338,844,47,63,49,...,0,0,0,0,0,0,0,0,0,0


In [15]:
percent_differences.head()

NameError: name 'percent_differences' is not defined

In [11]:
#NEED TO FORMAT THE IF CONDITION TO FIT THE YEAR FORMAT OF THE PERCENT_DIFFERENCES COLUMNS
def last_observed_year_after_2007(row):
    for i in range (2025,2007,-1):
        if row[f'{i}'] in (0,0.0): #increment through the proper column names
            if i+1<2026:
                return i+1
            else:
                return 2025
    return 2008

def get_alltime_percents(row):
    x = (totals.loc[row.name,"2025"]/totals.loc[0,"2025"])/0.09315068493 #0.09315068493 corrects for the fact that 2025 data represents 9.31% of the year so far.
    a = totals.loc[row.name,f'{earliest.iloc[row.name]}']
    b=totals.loc[0,f'{earliest.iloc[row.name]}']
    y =a/b
    return (100*((x-y)/y-1))


In [ ]:
totals.head()

In [165]:
differences.head(30)

,taxon_name,total_change,24-25,23-24,22-23,21-22,20-21,19-20,18-19,17-18,16-17,15-16,14-15,13-14,12-13,11-12,10-11,09-10,08-09
0,All Taxa,11669.0,-96519.0,36061.0,-30252.0,51836.0,49101.0,-430.0,-368.0,641.0,461.0,-54.0,84.0,674.0,-290.0,151.0,146.0,347.0,80.0
1,Chelonia mydas,105.0,-1097.0,212.0,-321.0,578.0,693.0,-45.0,-26.0,42.0,21.0,7.0,13.0,13.0,-5.0,8.0,-6.0,12.0,6.0
2,Paroaria coronata,123.0,-1194.0,352.0,-256.0,476.0,739.0,-27.0,-10.0,25.0,0.0,16.0,-13.0,8.0,-4.0,2.0,-7.0,14.0,2.0
3,Geopelia striata,167.0,-1153.0,373.0,-114.0,455.0,599.0,-31.0,9.0,15.0,3.0,1.0,-6.0,11.0,-5.0,3.0,2.0,6.0,-1.0
4,Metrosideros polymorpha,164.0,-1072.0,424.0,-368.0,566.0,605.0,-37.0,27.0,-2.0,-3.0,10.0,-1.0,12.0,-2.0,-7.0,7.0,3.0,2.0
5,Phelsuma laticauda,128.0,-1408.0,339.0,-139.0,494.0,797.0,-16.0,14.0,10.0,6.0,14.0,-2.0,14.0,0.0,0.0,0.0,6.0,-1.0
6,Anolis sagrei,95.0,-1078.0,373.0,-201.0,398.0,587.0,-22.0,3.0,5.0,21.0,5.0,-7.0,10.0,-1.0,1.0,1.0,2.0,-2.0
7,Acridotheres tristis,117.0,-1083.0,430.0,-266.0,491.0,539.0,-10.0,-9.0,19.0,-4.0,2.0,-3.0,1.0,6.0,-3.0,6.0,-1.0,2.0
8,Pluvialis fulva,169.0,-928.0,440.0,-144.0,457.0,340.0,-21.0,-4.0,18.0,-5.0,1.0,3.0,2.0,2.0,5.0,-7.0,9.0,1.0
9,Ardea ibis,101.0,-825.0,306.0,-177.0,312.0,477.0,-4.0,-24.0,30.0,-3.0,0.0,0.0,6.0,1.0,3.0,-4.0,2.0,1.0


I need to remove 2025 completely. I cannot extrapolate the data from the first month of the year to the entire year, especially if some taxa are seasonal etc. 

Percent_differences["total change"] method is correct. The percent differene method is too. the 24-25 and total change columns of differences is dumb.

% difference is wrong for all taxa because it gets 1/1-1. It does not need to be relative, and should use a different calculation.4


In PErcent Differences, Need to change all instances of -200 before 'inf' to be -100, due to the way the formula works.

Next up: graph all total changes on a graph. Include invasive or native color labels.



IMPLEMENT: only calculate percent change for taxa whose observations are above .5% of the annual median observations. Replace those under with NAN and then ignore them.




In [12]:
cols2 = ["taxon_name","total_change"]
for i in range(2,19): #until 2008, the release year of INaturalist, and the year that a significant amount of observations were first made.
    cols2.append(f"{cols[i][2:]}-{cols[i-1][2:]}")
differences = pd.DataFrame(columns = cols2)
differences["taxon_name"] = totals["taxon_name"]
differences.head()


#find the differences
for i in range(2,19):
    differences.iloc[:,i]=(totals.loc[:,cols[i-1]]-totals.loc[:,cols[i]])
differences.iloc[:,1] = totals.loc[:,"2025"]-totals.loc[:,"2008"]

#create another dataframe, percent_differences
percent_differences = pd.DataFrame(columns=differences.columns)
percent_differences["taxon_name"] = totals["taxon_name"]

for i in range(2,19):#Apply my formula from above to find percent differences across all taxa and years
    year_prop = (totals.loc[:,cols[i-1]]/totals.loc[0,cols[i-1]])
    prev_year_prop = (totals.loc[:,cols[i]]/totals.loc[0,cols[i]])
    percent_differences.iloc[:,i]=100*((year_prop-prev_year_prop)/prev_year_prop-1)

percent_differences.head(10)

for i in range (2,19):#Apply formula to find total_change column for all taxa
    earliest = totals.apply(last_observed_year_after_2007, axis=1)
percent_differences["total_change"] = percent_differences.apply(get_alltime_percents,axis=1)
percent_differences.head(100)



C:\Users\jcwvi\AppData\Local\Temp\ipykernel_27044\1860717842.py:16: RuntimeWarning: invalid value encountered in scalar divide
  return (100*((x-y)/y-1))


,taxon_name,total_change,24-25,23-24,22-23,21-22,20-21,19-20,18-19,17-18,16-17,15-16,14-15,13-14,12-13,11-12,10-11,09-10,08-09
0,All Taxa,873.529412,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0
1,Chelonia mydas,146.532004,-114.231725,-119.072686,-92.696725,-111.931012,-148.042571,-134.646904,-108.408128,-84.933418,-96.226667,-82.111,-73.162714,-118.89827,-85.322093,-83.043441,-134.034537,-104.482984,-70.695444
2,Paroaria coronata,571.410373,-112.479464,-108.997729,-87.850573,-119.027376,38.395154,-164.055797,-107.644862,-43.215063,-123.811983,140.600484,-166.922825,-111.072664,-96.789883,-101.979265,-144.029304,-36.256545,-86.858513
3,Geopelia striata,1088.195057,-83.254653,-107.039173,-73.425359,-113.517392,-6.105274,-167.479055,-51.524754,-62.268788,-108.57438,-88.934625,-133.845651,-81.430219,-111.534186,-90.840545,-97.387057,-36.256545,-139.388489
4,Metrosideros polymorpha,379.227407,-74.39463,-98.744481,-102.1238,-105.462783,-78.983556,-159.362257,29.151846,-129.882292,-130.738167,-51.268976,-109.431546,-93.287197,-82.045581,-142.821238,-74.065934,-130.533079,-101.225686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Sesuvium portulacastrum,824.529402,-51.529549,-117.727713,-87.139331,-118.345243,-50.81761,-13.743676,-122.227252,-87.310827,-123.811983,-148.169492,-105.493787,13.425606,-62.386511,-171.410619,51.868132,-145.418848,-119.184652
96,Haemorhous mexicanus,368.507413,-98.539251,-117.690217,-115.588492,-51.396082,-96.271619,-125.49747,-141.670439,-12.184711,-139.049587,318.305085,-181.098757,inf,-200.0,-171.410619,inf,-200.0,inf
97,Epipremnum aureum,1531.095237,-70.250282,-113.856249,-105.111093,-90.024012,-72.961871,-75.829117,-25.011317,-149.915923,inf,-200.0,-105.493787,inf,-200.0,inf,NaN,NaN,NaN
98,Phoebastria immutabilis,1412.797768,13.426273,-96.980788,-87.883233,-78.227345,-149.650943,-158.609706,-54.176098,-174.242475,244.430096,-75.60678,36.265533,-164.429066,-62.386511,inf,-200.0,-90.837696,inf


Graph INaturalists growth in observations per year? see where it started to boom in popoularity?

In [54]:
pd.set_option('display.max_rows', 500)  # Adjust to your preference
pd.set_option('display.max_columns', 50)

In [ ]:
print(totals.iloc[6])
print('h')

In [22]:
totals["2023"].iloc[0:1000].median()*0.005

0.145

In [20]:
totals.to_csv("totals.csv")

GRAPH IDEA: Faceted histgram invasive vs endemic/native of log of observations. One each year, then add a slider to slide between them.
x: log # observations
y: # of taxa with that amount of observations

In [26]:
totals.head(100)

,total_observations,establishment_means,2025,2024,2023,2022,2021,2020,2019,2018,...,1974,1973,1972,1971,1969,1961,1947,1946,1933,1893
taxon_name,,,,,,,,,,,,,,,,,,,,,
Chelonia mydas,5081,NaN,115,1212,1000,1321,743,50,95,121,...,0,0,0,0,1,0,0,0,0,0
Paroaria coronata,4676,NaN,128,1322,970,1226,750,11,38,48,...,0,0,0,0,0,0,0,0,0,0
Geopelia striata,4346,NaN,171,1324,951,1065,610,11,42,33,...,0,0,0,0,0,0,0,0,0,0
Metrosideros polymorpha,4399,NaN,173,1245,821,1189,623,18,55,28,...,0,0,2,0,0,0,0,0,0,0
Phelsuma laticauda,5353,NaN,130,1538,1199,1338,844,47,63,49,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Haemorhous mexicanus,777,NaN,22,196,159,267,89,3,5,10,...,0,0,0,0,0,0,0,0,0,0
Epipremnum aureum,776,NaN,30,209,162,242,109,3,3,2,...,0,0,0,0,0,0,0,0,0,0
Phoebastria immutabilis,772,NaN,51,216,140,177,72,5,15,12,...,0,0,0,0,0,0,0,0,0,0
